# Module 06 — Lab: Hand-rolled agents

We build:
1. A multi-tool research agent loop with a step budget.
2. A plan-then-act variant.
3. A simple subagent pattern.

We do NOT depend on the Agent SDK here so you understand the primitives.
See `06-agents/sdk_example.py` (optional) for the SDK version.

In [ ]:
import os, json, time
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv('../.env')
client = Anthropic()
MODEL = os.getenv('ANTHROPIC_MODEL', 'claude-sonnet-4-6')
FAST  = os.getenv('ANTHROPIC_FAST_MODEL', 'claude-haiku-4-5-20251001')

## 1. Tools — mocked search and a calculator

In [ ]:
FAKE_INDEX = {
    'population of tokyo': 'Tokyo metropolitan population: ~13.96 million (2024 estimate).',
    'population of paris': 'Paris commune population: ~2.10 million (2024).',
    'speed of light':      'c = 299,792,458 m/s in vacuum.',
    'capital of japan':    'Tokyo is the capital of Japan.',
}

def web_search(query: str) -> str:
    key = query.strip().lower()
    for k, v in FAKE_INDEX.items():
        if k in key:
            return v
    return 'no results'

def calculator(expression: str) -> str:
    import ast
    return str(eval(compile(ast.parse(expression, mode='eval'), '', 'eval'), {'__builtins__': {}}, {}))

TOOLS = [
    {'name':'web_search',
     'description':'Search a curated index. Returns one short snippet.',
     'input_schema':{'type':'object','properties':{'query':{'type':'string'}},'required':['query']}},
    {'name':'calculator',
     'description':'Evaluate an arithmetic expression. Numbers and + - * / ** only.',
     'input_schema':{'type':'object','properties':{'expression':{'type':'string'}},'required':['expression']}},
]
TOOL_FNS = {'web_search': web_search, 'calculator': calculator}

## 2. Agent loop with step budget and cost tracking

In [ ]:
def run_tools(content):
    out = []
    for b in content:
        if b.type != 'tool_use': continue
        try:
            r = TOOL_FNS[b.name](**b.input)
            out.append({'type':'tool_result','tool_use_id': b.id,'content': str(r)})
        except Exception as e:
            out.append({'type':'tool_result','tool_use_id': b.id,'content': f'ERROR: {e}', 'is_error': True})
    return out

def agent(goal, system, max_steps=8, model=MODEL):
    messages = [{'role':'user','content': goal}]
    total_in = total_out = 0
    for step in range(max_steps):
        r = client.messages.create(
            model=model, max_tokens=2048,
            system=system, tools=TOOLS, messages=messages,
        )
        total_in  += r.usage.input_tokens
        total_out += r.usage.output_tokens
        messages.append({'role':'assistant','content': r.content})

        for b in r.content:
            if b.type == 'tool_use':  print(f'[step {step}] CALL {b.name}({b.input})')
            elif b.type == 'text':    print(f'[step {step}] TEXT: {b.text[:160]}')

        if r.stop_reason == 'end_turn':
            print(f'\nDONE in {step+1} steps. tokens in={total_in} out={total_out}')
            return ''.join(b.text for b in r.content if b.type == 'text')
        if r.stop_reason == 'tool_use':
            messages.append({'role':'user','content': run_tools(r.content)})
            continue
        raise RuntimeError(f'unexpected stop: {r.stop_reason}')
    print('!! step budget exhausted')
    return ''.join(b.text for b in r.content if b.type == 'text')

SYSTEM = ('You are a research assistant. Use tools when you need facts you don\'t know. '
          'When you have enough information, answer concisely with no tags.')

answer = agent(
    'What is the population of Tokyo, divided by 1000? Show me an integer.',
    SYSTEM, max_steps=6,
)
print('\nFINAL:', answer)

## 3. Plan-then-act variant

Force the model to emit a `<plan>` block before its first tool call.

In [ ]:
PLANNING_SYSTEM = ('You are a research assistant. Before any tool call, emit '
                   '<plan>brief 2-3 step plan</plan>. Then act. Final answer has no tags.')

agent('What is the speed of light times 60 (in m/s * s)?', PLANNING_SYSTEM, max_steps=6)

## 4. Subagent pattern

Parent agent uses Sonnet. It delegates web-search to a cheaper Haiku subagent that returns only the summary.

In [ ]:
def search_subagent(query: str) -> str:
    """Cheap Haiku-powered subagent that runs the web_search tool and synthesizes a one-liner."""
    r = client.messages.create(
        model=FAST, max_tokens=400,
        system='You answer questions using the web_search tool only. Reply in <=20 words.',
        tools=[TOOLS[0]],
        messages=[{'role':'user','content': query}],
    )
    # one-pass tool loop for simplicity
    if r.stop_reason == 'tool_use':
        results = run_tools(r.content)
        r2 = client.messages.create(
            model=FAST, max_tokens=200,
            system='Reply in <=20 words.',
            tools=[TOOLS[0]],
            messages=[
                {'role':'user','content': query},
                {'role':'assistant','content': r.content},
                {'role':'user','content': results},
            ],
        )
        return ''.join(b.text for b in r2.content if b.type == 'text')
    return ''.join(b.text for b in r.content if b.type == 'text')

# Expose the subagent as a tool to the parent
PARENT_TOOLS = [
    {'name':'research',
     'description':'Delegate a research question to a fast research subagent. Returns a short answer.',
     'input_schema':{'type':'object','properties':{'question':{'type':'string'}},'required':['question']}},
    TOOLS[1],  # calculator
]

def run_parent_tools(content):
    out = []
    for b in content:
        if b.type != 'tool_use': continue
        if b.name == 'research':
            r = search_subagent(b.input['question'])
        else:
            r = TOOL_FNS[b.name](**b.input)
        out.append({'type':'tool_result','tool_use_id': b.id,'content': str(r)})
    return out

def parent_agent(goal, max_steps=6):
    messages = [{'role':'user','content': goal}]
    for step in range(max_steps):
        r = client.messages.create(
            model=MODEL, max_tokens=1024,
            system='You orchestrate. Use the research tool for facts; use calculator for math.',
            tools=PARENT_TOOLS, messages=messages,
        )
        messages.append({'role':'assistant','content': r.content})
        if r.stop_reason == 'end_turn':
            return ''.join(b.text for b in r.content if b.type == 'text')
        if r.stop_reason == 'tool_use':
            messages.append({'role':'user','content': run_parent_tools(r.content)})
            continue
        raise RuntimeError('bad stop')
    return ''.join(b.text for b in r.content if b.type == 'text')

print(parent_agent('How many people live in Paris and Tokyo combined? Give the integer in millions.'))